# Detección de EPP en manufactura

Notebook simplificado para usar un modelo YOLOv8 ya entrenado.

> El modelo es una herramienta de apoyo y debe validarse con datos reales de la planta antes de usarse en decisiones de seguridad.


## Contexto teórico del caso: seguridad en manufactura

Las áreas de manufactura combinan personas, maquinaria, vehículos internos, herramientas y materiales. Esta interacción puede generar riesgos como golpes, atrapamientos, proyección de partículas, contacto con sustancias y circulación en zonas restringidas. Los equipos de protección personal (EPP) funcionan como una barrera adicional cuando los controles de ingeniería y administrativos no eliminan completamente el peligro.

La verificación del EPP suele depender de recorridos, supervisión y auditorías. Aunque estos mecanismos son indispensables, pueden tener cobertura limitada y no observar continuamente todos los puntos de la operación. La visión artificial permite analizar imágenes o video de cámaras para identificar condiciones visibles de cumplimiento, como el uso de casco, chaleco o mascarilla, y generar una alerta para revisión humana.

En este caso, el problema se formula como **detección de objetos y posibles incumplimientos de EPP** en imágenes de una operación de manufactura. El sistema no determina por sí mismo que ocurrió un accidente ni sustituye un análisis de riesgos; identifica señales visuales que pueden apoyar la prevención.


## Fundamento de la técnica: deep learning y YOLOv8

El deep learning utiliza redes neuronales con múltiples capas para aprender representaciones directamente a partir de datos. En visión artificial, una red convolucional aprende patrones visuales jerárquicos: bordes y texturas en capas iniciales, y formas más complejas en capas posteriores. Para que el aprendizaje sea posible, el modelo necesita imágenes representativas y etiquetas que indiquen la ubicación y la clase de cada objeto.

YOLOv8 es una familia de modelos de detección de objetos que procesa una imagen en una sola pasada de la red. Para cada objeto localizado, produce una **caja delimitadora**, una **clase** y un **nivel de confianza**. En este notebook se utiliza un modelo previamente entrenado (`best.pt`), por lo que la etapa ejecutada corresponde a la **inferencia**: aplicar el conocimiento aprendido a imágenes o videos nuevos.

La ventaja de utilizar un modelo ya entrenado es la rapidez de operación y la posibilidad de probar el concepto sin instalar un entorno de entrenamiento ni descargar un conjunto completo de datos. Sin embargo, el modelo conserva los sesgos y limitaciones de las imágenes con las que fue entrenado.


## Interpretación de resultados y criterios de evaluación

Cada detección debe interpretarse considerando su confianza y el contexto de la escena. Un umbral alto reduce algunas falsas alarmas, pero puede omitir incumplimientos; un umbral bajo puede detectar más casos, aunque también producir más falsos positivos. Por ello, la calibración debe realizarse con imágenes reales de la planta.

Para evaluar la técnica se pueden utilizar:

- **Precisión:** proporción de detecciones reportadas que son correctas.
- **Recall o sensibilidad:** proporción de incumplimientos reales que el modelo logra detectar.
- **Falsos positivos:** alertas en las que sí había EPP o la detección fue incorrecta.
- **Falsos negativos:** incumplimientos que el sistema no detectó.
- **Tiempo de inferencia:** rapidez con la que analiza una imagen o cuadro de video.

En seguridad industrial, los falsos negativos son especialmente importantes porque un incumplimiento no detectado puede quedar sin revisión. Por eso el resultado del modelo debe integrarse con supervisión humana, protocolos de respuesta, protección de datos y validación periódica del desempeño.


## 1. Instalar y cargar el modelo (Opción 1) accesar a Hugging Face.

In [ ]:
!pip install -q ultralytics

from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, Video
from google.colab import files
from ultralytics import YOLO

BASE_DIR = Path('/content/ppe')
BASE_DIR.mkdir(exist_ok=True)
MODEL_PATH = BASE_DIR / 'best.pt'

# Descarga directa del peso entrenado; no es necesario entrar ni clonar un repo.
MODEL_URL = 'https://huggingface.co/Hansung-Cho/yolov8-ppe-detection/resolve/main/best.pt'
if not MODEL_PATH.exists():
    !wget -q --show-progress -O {MODEL_PATH} {MODEL_URL}

model = YOLO(str(MODEL_PATH))
print('Modelo listo.')
print('Clases:', model.names)


## 2. Opción 2. Subir archivos (Best.pt) e Imagenes a evaluar. Si usas el paso 1. Es suficiente solo ingresar las imagenes.

In [ ]:
# Carga en una sola ventana el modelo best.pt (opcional), una o varias imágenes y/o videos.
# Imágenes: JPG, JPEG, PNG, BMP, WEBP. Videos: MP4, MOV, AVI, MKV.
# Puedes seleccionar, por ejemplo: best.pt + imagen1.jpg + imagen2.jpg.
uploaded = files.upload()
input_paths = []
for filename, content in uploaded.items():
    path = BASE_DIR / filename
    path.write_bytes(content)
    input_paths.append(path)
print('Archivos cargados:', [path.name for path in input_paths])

# Si se cargó un modelo, reemplaza el modelo descargado automáticamente.
uploaded_models = [path for path in input_paths if path.suffix.lower() == '.pt']
if uploaded_models:
    model = YOLO(str(uploaded_models[0]))
    print('Se usará el modelo cargado:', uploaded_models[0].name)
else:
    print('Se usará el modelo automático:', MODEL_PATH.name)


## 3. Analizar imágenes y videos

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv'}
images = [path for path in input_paths if path.suffix.lower() in IMAGE_EXTENSIONS]
videos = [path for path in input_paths if path.suffix.lower() in VIDEO_EXTENSIONS]

if images:
    image_results = model.predict(
        source=[str(path) for path in images],
        conf=0.40,
        save=True,
        project=str(BASE_DIR / 'results'),
        name='images',
        exist_ok=True,
        verbose=False,
    )
    fig, axes = plt.subplots(1, len(image_results), figsize=(7 * len(image_results), 6), squeeze=False)
    for axis, result, original in zip(axes.flat, image_results, images):
        axis.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
        axis.set_title(original.name)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

if videos:
    video_results = model.predict(
        source=str(videos[0]),
        conf=0.40,
        save=True,
        project=str(BASE_DIR / 'results'),
        name='videos',
        exist_ok=True,
        verbose=False,
    )
    video_dir = BASE_DIR / 'results' / 'videos'
    output_video = next((path for path in video_dir.iterdir() if path.suffix.lower() in VIDEO_EXTENSIONS), None)
    if output_video is None:
        raise FileNotFoundError('No se generó el video anotado.')
    print('Video generado:', output_video)
    display(Video(str(output_video), embed=True, width=900))
    files.download(str(output_video))

if not images and not videos:
    print('No se cargaron archivos compatibles.')


## Nota de uso

El modelo fue entrenado con imágenes de ambientes tipo construcción. En manufactura puede presentar errores por iluminación, uniformes, cámaras y tipos de EPP distintos. Se recomienda probarlo con una muestra representativa de la planta y revisar manualmente las alertas.

Las imagenes son ficticias. Son para fines educativos.


## Fuente y resultados del modelo entrenado

El archivo `best.pt` corresponde a un modelo YOLOv8n ajustado para detectar equipo de protección personal en escenas tipo construcción y seguridad industrial. La ficha pública del modelo reporta las siguientes métricas de validación:

- **mAP@0.50:** 0.744
- **mAP@0.50:0.95:** 0.436
- **Precisión:** 0.831
- **Recall:** 0.685

Entre sus clases se encuentran persona, casco, ausencia de casco, chaleco, ausencia de chaleco y mascarilla. Estos resultados pertenecen al conjunto de validación original del modelo; no representan todavía el desempeño específico en la planta. La fuente documental es el [model card de Hugging Face](https://huggingface.co/Hansung-Cho/yolov8-ppe-detection).

En este notebook, las predicciones sobre archivos propios se consideran resultados de inferencia. Para convertirlas en evidencia de desempeño, deben compararse posteriormente contra una revisión humana o etiquetas reales.


## Registro de resultados de inferencia

In [ ]:
# Genera un CSV con las detecciones encontradas en las imágenes cargadas.
import pandas as pd
from datetime import datetime

rows = []
if 'image_results' in globals():
    for original, result in zip(images, image_results):
        if result.boxes is not None and len(result.boxes) > 0:
            for cls_id, confidence in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist()):
                rows.append({
                    'fecha_hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'archivo': original.name,
                    'tipo': 'imagen',
                    'deteccion': model.names[int(cls_id)],
                    'confianza': round(float(confidence), 4),
                    'posible_incumplimiento': model.names[int(cls_id)].lower().startswith(('no-', 'no_')),
                })
        else:
            rows.append({
                'fecha_hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'archivo': original.name,
                'tipo': 'imagen',
                'deteccion': 'sin detecciones',
                'confianza': None,
                'posible_incumplimiento': False,
            })

if rows:
    results_df = pd.DataFrame(rows)
    csv_path = BASE_DIR / 'resultados_inferencia_ppe.csv'
    results_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    display(results_df)
    print('Reporte guardado en:', csv_path)
    files.download(str(csv_path))
else:
    print('No hay detecciones de imágenes para registrar. El video se entrega como evidencia visual.')


### Cómo interpretar el reporte

La columna **confianza** indica qué tan seguro está el modelo de su predicción; no equivale por sí sola a una probabilidad calibrada. La columna **posible_incumplimiento** identifica clases cuyo nombre indica ausencia de EPP. Cada registro debe revisarse con el contexto de la escena y, para una evaluación formal, compararse con una etiqueta humana.
